<a href="https://colab.research.google.com/github/evildead23151/SparseRNN-Systems/blob/main/SparseRNN_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# =========================
# ENVIRONMENT SETUP (FIXED)
# =========================

import torch
import numpy as np
import random

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

# 🔥 IMPORTANT: enable fast kernels
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


##Load Tiny Shakespeare Dataset

In [32]:
# =========================
# DOWNLOAD DATA
# =========================

import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

response = requests.get(url)
text = response.text

print("Dataset length:", len(text))
print("Sample:\n", text[:500])

Dataset length: 1115394
Sample:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [33]:
# =========================
# TOKENIZATION
# =========================

# Unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocab size:", vocab_size)

# Mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# Encode full dataset (IMPORTANT: contiguous tensor)
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

print("Encoded data shape:", data.shape)

Vocab size: 65
Encoded data shape: torch.Size([1115394])


In [34]:
# =========================
# BATCH PREPARATION
# =========================

batch_size = 64
seq_len = 128

# Total usable tokens
num_tokens = data.size(0)
num_batches = num_tokens // (batch_size * seq_len)

# Trim excess
data = data[:num_batches * batch_size * seq_len]

# Reshape → (batch_size, total_seq)
data = data.view(batch_size, -1)

print("Data reshaped:", data.shape)

Data reshaped: torch.Size([64, 17408])


In [35]:
# =========================
# SEQUENCE ITERATOR
# =========================

def get_batch(data, i, seq_len):
    """
    Returns:
    x: (batch_size, seq_len)
    y: (batch_size, seq_len)
    """
    x = data[:, i:i+seq_len]
    y = data[:, i+1:i+seq_len+1]
    return x, y

In [36]:
# =========================
# MOVE TO GPU
# =========================

data = data.to(device, non_blocking=True)

print("Data device:", data.device)

Data device: cuda:0


In [37]:
# =========================
# SYNTHETIC DATA
# =========================

def generate_synthetic_data(num_sequences=10000, seq_len=128, input_dim=256):
    data = torch.randn(num_sequences, seq_len, input_dim)

    # Add temporal correlation
    for t in range(1, seq_len):
        data[:, t] = 0.8 * data[:, t-1] + 0.2 * data[:, t]

    targets = torch.roll(data, shifts=-1, dims=1)

    return data, targets

In [38]:
# =========================
# SYNTHETIC PREP
# =========================

syn_data, syn_targets = generate_synthetic_data()

syn_data = syn_data.to(device)
syn_targets = syn_targets.to(device)

print("Synthetic data:", syn_data.shape)

Synthetic data: torch.Size([10000, 128, 256])


In [39]:
# =========================
# SANITY CHECK
# =========================

i = 0
x, y = get_batch(data, i, seq_len)

print("x shape:", x.shape)
print("y shape:", y.shape)

# decode sample
sample = x[0][:50].cpu().numpy()
decoded = ''.join([itos[int(i)] for i in sample])
print("Decoded sample:\n", decoded)

x shape: torch.Size([64, 128])
y shape: torch.Size([64, 128])
Decoded sample:
 First Citizen:
Before we proceed any further, hear


##MODEL

In [47]:
# =========================
# LSTM BASELINE MODEL (FIXED)
# =========================

import torch.nn as nn

class LSTMBaseline(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, x, hidden):
        x = self.embedding(x)
        x = x.contiguous()   # 🔥 IMPORTANT FIX

        out, hidden = self.lstm(x, hidden)
        return out, hidden

In [41]:
# =========================
# HIDDEN STATE INIT
# =========================

def init_hidden(batch_size, hidden_size, num_layers, device):
    h0 = torch.zeros(num_layers, batch_size, hidden_size, device=device)
    c0 = torch.zeros(num_layers, batch_size, hidden_size, device=device)
    return (h0, c0)

In [42]:
# =========================
# TRAIN SETUP
# =========================

hidden_size = 512
num_layers = 2

model = LSTMBaseline(vocab_size, hidden_size, num_layers).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [43]:
# =========================
# CLEAN CUDA TIMING
# =========================

def measure_forward_time(model, data, seq_len, steps=50):
    model.eval()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    timings = []

    # 🔥 FIXED INPUT (NO MOVING WINDOW)
    x, _ = get_batch(data, 0, seq_len)

    hidden = init_hidden(batch_size, hidden_size, num_layers, device)

    # Warmup
    for _ in range(10):
        out, hidden = model(x, hidden)
        hidden = (hidden[0].detach(), hidden[1].detach())

    torch.cuda.synchronize()

    # Measure
    for _ in range(steps):
        starter.record()

        out, hidden = model(x, hidden)

        ender.record()
        torch.cuda.synchronize()

        timings.append(starter.elapsed_time(ender))

        hidden = (hidden[0].detach(), hidden[1].detach())

    return sum(timings) / len(timings)

In [44]:
avg_time = measure_forward_time(model, data, seq_len)
print(f"Average forward pass time: {avg_time:.3f} ms")

Average forward pass time: 27.824 ms


In [45]:
# =========================
# DEBUG CHECK (RUN THIS)
# =========================

print("---- CUDA CHECK ----")
print("device:", device)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0))

print("\n---- CUDNN ----")
print("enabled:", torch.backends.cudnn.enabled)
print("benchmark:", torch.backends.cudnn.benchmark)

print("\n---- INPUT CHECK ----")
x, _ = get_batch(data, 0, seq_len)
print("x dtype:", x.dtype)
print("x device:", x.device)

print("\n---- EMBEDDING CHECK ----")
emb = model.embedding(x)
print("embedding shape:", emb.shape)
print("embedding dtype:", emb.dtype)
print("embedding contiguous:", emb.is_contiguous())

---- CUDA CHECK ----
device: cuda
cuda available: True
gpu: Tesla T4

---- CUDNN ----
enabled: True
benchmark: True

---- INPUT CHECK ----
x dtype: torch.int64
x device: cuda:0

---- EMBEDDING CHECK ----
embedding shape: torch.Size([64, 128, 512])
embedding dtype: torch.float32
embedding contiguous: True


In [48]:
# =========================
# PURE LSTM TEST (NO EMBEDDING)
# =========================

lstm = torch.nn.LSTM(512, 512, 2, batch_first=True).to(device)

x = torch.randn(batch_size, seq_len, 512, device=device)

h0 = torch.zeros(2, batch_size, 512, device=device)
c0 = torch.zeros(2, batch_size, 512, device=device)

starter = torch.cuda.Event(enable_timing=True)
ender = torch.cuda.Event(enable_timing=True)

# Warmup
for _ in range(10):
    out, _ = lstm(x, (h0, c0))

torch.cuda.synchronize()

# Measure
starter.record()
out, _ = lstm(x, (h0, c0))
ender.record()

torch.cuda.synchronize()

print("PURE LSTM time:", starter.elapsed_time(ender), "ms")

PURE LSTM time: 28.409536361694336 ms


In [57]:
# =========================
# MANUAL LSTM (GPU-READY LAYOUT)
# =========================

import torch
import torch.nn as nn

class ManualLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        # Separate projections (important for CUDA mapping)
        self.W_x = nn.Linear(input_size, 4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

    def forward(self, x, hidden):
        """
        x: (B, T, D)
        hidden: (h, c) each (B, H)
        """

        B, T, D = x.shape
        H = self.hidden_size

        h, c = hidden

        outputs = []

        # 🔥 Input projection (one big GEMM)
        x_proj = self.W_x(x)   # (B, T, 4H)

        # 🔥 CHANGE MEMORY LAYOUT FOR GPU
        x_proj = x_proj.permute(1, 0, 2).contiguous()   # (T, B, 4H)

        # Time loop (sequential dependency)
        for t in range(T):
            # x_proj[t] is now contiguous (B, 4H)
            gates = x_proj[t] + self.W_h(h)   # (B, 4H)

            # Split gates
            i, f, g, o = torch.chunk(gates, 4, dim=1)

            # Activations
            i = torch.sigmoid(i)
            f = torch.sigmoid(f)
            o = torch.sigmoid(o)
            g = torch.tanh(g)

            # Cell update
            c = f * c + i * g
            h = o * torch.tanh(c)

            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)   # (B, T, H)

        return outputs, (h, c)

In [58]:
# =========================
# MANUAL LSTM MODEL
# =========================

class ManualLSTMModel(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = ManualLSTM(hidden_size, hidden_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        x = x.contiguous()

        out, hidden = self.lstm(x, hidden)
        return out, hidden

In [59]:
# =========================
# INIT MANUAL MODEL
# =========================

manual_model = ManualLSTMModel(vocab_size, hidden_size).to(device)

def init_hidden_manual(batch_size, hidden_size, device):
    h = torch.zeros(batch_size, hidden_size, device=device)
    c = torch.zeros(batch_size, hidden_size, device=device)
    return (h, c)

In [60]:
# =========================
# TEST MANUAL LSTM
# =========================

x, _ = get_batch(data, 0, seq_len)
hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = manual_model(x, hidden)

print("Output shape:", out.shape)

Output shape: torch.Size([64, 128, 512])


##CUDA-STYLE PIPELINE

In [64]:
# =========================
# CUDA-STYLE DENSE LSTM (FINAL, FUSED VERSION)
# =========================

import torch
import torch.nn as nn

# 🔥 FUSED GATE FUNCTION (CUDA KERNEL SIMULATION)
def fused_lstm_step(gates, c_prev):
    """
    gates: (B, 4H)
    c_prev: (B, H)
    """

    B, fourH = gates.shape
    H = fourH // 4

    i = torch.sigmoid(gates[:, :H])
    f = torch.sigmoid(gates[:, H:2*H])
    g = torch.tanh(gates[:, 2*H:3*H])
    o = torch.sigmoid(gates[:, 3*H:])

    c = f * c_prev + i * g
    h = o * torch.tanh(c)

    return h, c


class CUDADenseLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        # Maps directly to cuBLAS GEMMs
        self.W_x = nn.Linear(input_size, 4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

    def forward(self, x, hidden):
        """
        x: (B, T, D)
        hidden: (h, c)
        """

        B, T, D = x.shape
        H = self.hidden_size

        h, c = hidden

        # =========================
        # STEP 1: BIG GEMM (cuBLAS equivalent)
        # =========================
        x_reshaped = x.reshape(B * T, D)          # (B*T, D)
        x_proj = self.W_x(x_reshaped)             # (B*T, 4H)

        # =========================
        # STEP 2: RESHAPE (GPU-FRIENDLY)
        # =========================
        x_proj = x_proj.view(T, B, 4 * H).contiguous()   # (T, B, 4H)

        outputs = []

        # =========================
        # STEP 3: TIME LOOP
        # =========================
        for t in range(T):

            # SMALL GEMM (cuBLAS)
            h_proj = self.W_h(h)   # (B, 4H)

            # ADD + FUSED COMPUTE INPUT
            gates = x_proj[t] + h_proj   # (B, 4H)

            # 🔥 FUSED GATE OPERATION (CRITICAL)
            h, c = fused_lstm_step(gates, c)

            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)   # (B, T, H)

        return outputs, (h, c)

In [65]:
# =========================
# TEST CUDA-STYLE LSTM
# =========================

cuda_style_model = CUDADenseLSTM(hidden_size, hidden_size).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)   # reuse embedding

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = cuda_style_model(x, hidden)

print("CUDA-style output shape:", out.shape)

CUDA-style output shape: torch.Size([64, 128, 512])


In [66]:
# =========================
# FUSED GATE (SIMULATION)
# =========================

def fused_lstm_step(gates, c_prev):
    """
    gates: (B, 4H)
    c_prev: (B, H)
    """

    B, fourH = gates.shape
    H = fourH // 4

    i = torch.sigmoid(gates[:, :H])
    f = torch.sigmoid(gates[:, H:2*H])
    g = torch.tanh(gates[:, 2*H:3*H])
    o = torch.sigmoid(gates[:, 3*H:])

    c = f * c_prev + i * g
    h = o * torch.tanh(c)

    return h, c

In [68]:
%%writefile lstm_kernel.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <torch/extension.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

__device__ __forceinline__ float sigmoidf_fast(float x) {
    return 1.f / (1.f + expf(-x));
}

// Each block = one batch element (b)
// Each thread = one hidden index (j)
__global__ void lstm_fused_kernel(
    const float* __restrict__ gates,   // (B, 4H)
    const float* __restrict__ c_prev,  // (B, H)
    float* __restrict__ h_out,         // (B, H)
    float* __restrict__ c_out,         // (B, H)
    int B,
    int H
){
    int b = blockIdx.x;
    int j = threadIdx.x;

    if (b >= B || j >= H) return;

    int baseH = b * H;
    int base4H = b * 4 * H;

    // gate slices
    float gi = gates[base4H + j];
    float gf = gates[base4H + j + H];
    float gg = gates[base4H + j + 2*H];
    float go = gates[base4H + j + 3*H];

    float i = sigmoidf_fast(gi);
    float f = sigmoidf_fast(gf);
    float o = sigmoidf_fast(go);
    float g = tanhf(gg);

    float c_prev_val = c_prev[baseH + j];

    float c = f * c_prev_val + i * g;
    float h = o * tanhf(c);

    c_out[baseH + j] = c;
    h_out[baseH + j] = h;
}

std::vector<torch::Tensor> lstm_fused_forward(
    torch::Tensor gates,   // (B, 4H)
    torch::Tensor c_prev   // (B, H)
){
    CHECK_INPUT(gates);
    CHECK_INPUT(c_prev);

    auto B = gates.size(0);
    auto fourH = gates.size(1);
    auto H = fourH / 4;

    auto h_out = torch::empty({B, H}, gates.options());
    auto c_out = torch::empty({B, H}, gates.options());

    const int threads = 256;
    const int blocks = B;

    lstm_fused_kernel<<<blocks, threads>>>(
        gates.data_ptr<float>(),
        c_prev.data_ptr<float>(),
        h_out.data_ptr<float>(),
        c_out.data_ptr<float>(),
        B,
        H
    );

    return {h_out, c_out};
}

Writing lstm_kernel.cu


In [69]:
%%writefile lstm_binding.cpp
#include <torch/extension.h>

// forward declaration
std::vector<torch::Tensor> lstm_fused_forward(
    torch::Tensor gates,
    torch::Tensor c_prev
);

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &lstm_fused_forward, "LSTM fused forward (CUDA)");
}

Writing lstm_binding.cpp


In [71]:
# =========================
# INSTALL NINJA (REQUIRED)
# =========================

!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.5 MB/s eta 0:00:00


In [72]:
# =========================
# COMPILE CUDA EXTENSION (FIXED)
# =========================

from torch.utils.cpp_extension import load
import os

# Optional: clear cache if rebuild issues
!rm -rf /root/.cache/torch_extensions

lstm_cuda = load(
    name="lstm_cuda",
    sources=["lstm_binding.cpp", "lstm_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"]
)

print("CUDA extension compiled successfully")

CUDA extension compiled successfully


In [73]:
# =========================
# TEST CUDA KERNEL (ISOLATED)
# =========================

B = batch_size
H = hidden_size

gates = torch.randn(B, 4*H, device=device, dtype=torch.float32).contiguous()
c_prev = torch.randn(B, H, device=device, dtype=torch.float32).contiguous()

h_out, c_out = lstm_cuda.forward(gates, c_prev)

print("h_out shape:", h_out.shape)
print("c_out shape:", c_out.shape)

h_out shape: torch.Size([64, 512])
c_out shape: torch.Size([64, 512])


In [74]:
# =========================
# FINAL MODEL TEST
# =========================

cuda_model = CUDADenseLSTM(hidden_size, hidden_size).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = cuda_model(x, hidden)

print("final output:", out.shape)

final output: torch.Size([64, 128, 512])


In [75]:
# =========================
# BLOCK SPARSE MASK + WEIGHT
# =========================

def create_block_mask(out_dim, in_dim, block_size=16, sparsity=0.5):
    """
    Creates block-level mask
    """

    assert out_dim % block_size == 0
    assert in_dim % block_size == 0

    num_blocks_row = out_dim // block_size
    num_blocks_col = in_dim // block_size

    mask = torch.ones(num_blocks_row, num_blocks_col)

    # randomly zero blocks
    total_blocks = num_blocks_row * num_blocks_col
    num_zero = int(total_blocks * sparsity)

    idx = torch.randperm(total_blocks)[:num_zero]

    for i in idx:
        r = i // num_blocks_col
        c = i % num_blocks_col
        mask[r, c] = 0

    return mask


def expand_block_mask(mask, block_size):
    """
    Expand block mask to full matrix
    """
    return mask.repeat_interleave(block_size, dim=0).repeat_interleave(block_size, dim=1)

In [82]:
# =========================
# TRUE BLOCK-SPARSE LSTM (FULL, SELF-CONTAINED)
# =========================

import torch
import torch.nn as nn

# -------------------------
# BLOCK MASK CREATION
# -------------------------
def create_block_mask(out_dim, in_dim, block_size=16, sparsity=0.5):
    assert out_dim % block_size == 0
    assert in_dim % block_size == 0

    num_blocks_row = out_dim // block_size
    num_blocks_col = in_dim // block_size

    mask = torch.ones(num_blocks_row, num_blocks_col)

    total_blocks = num_blocks_row * num_blocks_col
    num_zero = int(total_blocks * sparsity)

    idx = torch.randperm(total_blocks)[:num_zero]

    for i in idx:
        r = i // num_blocks_col
        c = i % num_blocks_col
        mask[r, c] = 0

    return mask


# -------------------------
# GET ACTIVE BLOCKS
# -------------------------
def get_active_blocks(mask):
    active_blocks = []
    rows, cols = mask.shape

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] == 1:
                active_blocks.append((i, j))

    return active_blocks


# -------------------------
# TRUE BLOCK-SPARSE MATMUL
# -------------------------
def block_sparse_matmul(h, W, mask, block_size):
    """
    h: (B, H)
    W: (4H, H)
    mask: (4H/block, H/block)
    """

    B, H = h.shape
    out_dim = W.shape[0]

    result = torch.zeros(B, out_dim, device=h.device)

    active_blocks = get_active_blocks(mask)

    for (i, j) in active_blocks:
        row_start = i * block_size
        col_start = j * block_size

        W_block = W[row_start:row_start+block_size,
                    col_start:col_start+block_size]

        h_block = h[:, col_start:col_start+block_size]

        result[:, row_start:row_start+block_size] += torch.matmul(
            h_block,
            W_block.t()
        )

    return result


# -------------------------
# FUSED LSTM STEP
# -------------------------
def fused_lstm_step(gates, c_prev):
    B, fourH = gates.shape
    H = fourH // 4

    i = torch.sigmoid(gates[:, :H])
    f = torch.sigmoid(gates[:, H:2*H])
    g = torch.tanh(gates[:, 2*H:3*H])
    o = torch.sigmoid(gates[:, 3*H:])

    c = f * c_prev + i * g
    h = o * torch.tanh(c)

    return h, c


# -------------------------
# TRUE BLOCK-SPARSE LSTM
# -------------------------
class TrueBlockSparseLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, sparsity=0.5, block_size=16):
        super().__init__()

        self.hidden_size = hidden_size
        self.block_size = block_size

        self.W_x = nn.Linear(input_size, 4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

        # block-level mask
        block_mask = create_block_mask(
            4 * hidden_size,
            hidden_size,
            block_size,
            sparsity
        )

        self.register_buffer("block_mask", block_mask)

    def forward(self, x, hidden):
        B, T, D = x.shape
        H = self.hidden_size

        h, c = hidden

        # input GEMM
        x_reshaped = x.reshape(B * T, D)
        x_proj = self.W_x(x_reshaped)
        x_proj = x_proj.view(T, B, 4 * H).contiguous()

        outputs = []

        for t in range(T):
            # TRUE sparse compute
            h_proj = block_sparse_matmul(
                h,
                self.W_h.weight,
                self.block_mask,
                self.block_size
            ) + self.W_h.bias

            gates = x_proj[t] + h_proj

            h, c = fused_lstm_step(gates, c)

            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)

        return outputs, (h, c)


# -------------------------
# TEST
# -------------------------
sparse_model = TrueBlockSparseLSTM(
    input_size=hidden_size,
    hidden_size=hidden_size,
    sparsity=0.5,
    block_size=16
).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = sparse_model(x, hidden)

print("true sparse output:", out.shape)

true sparse output: torch.Size([64, 128, 512])


In [83]:
# =========================
# TEST BLOCK-SPARSE MODEL
# =========================

sparse_model = BlockSparseLSTM(
    input_size=hidden_size,
    hidden_size=hidden_size,
    sparsity=0.5,
    block_size=16
).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = sparse_model(x, hidden)

print("sparse output:", out.shape)

sparse output: torch.Size([64, 128, 512])


In [84]:
# =========================
# BLOCK INDEX EXTRACTION
# =========================

def get_active_blocks(mask, block_size):
    """
    Returns list of active (row_block, col_block)
    """

    block_rows, block_cols = mask.shape
    active_blocks = []

    for i in range(block_rows):
        for j in range(block_cols):
            if mask[i, j] == 1:
                active_blocks.append((i, j))

    return active_blocks

In [85]:
# =========================
# TRUE BLOCK-SPARSE MATMUL
# =========================

def block_sparse_matmul(h, W, mask, block_size):
    """
    h: (B, H)
    W: (4H, H)
    mask: block mask
    """

    B, H = h.shape
    out_dim = W.shape[0]

    result = torch.zeros(B, out_dim, device=h.device)

    active_blocks = get_active_blocks(mask, block_size)

    for (i, j) in active_blocks:
        # block indices
        row_start = i * block_size
        col_start = j * block_size

        # slices
        W_block = W[row_start:row_start+block_size,
                    col_start:col_start+block_size]

        h_block = h[:, col_start:col_start+block_size]

        # compute only active block
        result[:, row_start:row_start+block_size] += torch.matmul(
            h_block,
            W_block.t()
        )

    return result

In [86]:
sparse_model = TrueBlockSparseLSTM(
    input_size=hidden_size,
    hidden_size=hidden_size,
    sparsity=0.5
).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = sparse_model(x, hidden)

print("true sparse output:", out.shape)

true sparse output: torch.Size([64, 128, 512])


In [87]:
%%writefile sparse_lstm_kernel.cu

#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void block_sparse_matmul_kernel(
    const float* __restrict__ h,        // (B, H)
    const float* __restrict__ W,        // (4H, H)
    const int* __restrict__ row_idx,    // block rows
    const int* __restrict__ col_idx,    // block cols
    float* __restrict__ out,            // (B, 4H)
    int B,
    int H,
    int block_size,
    int num_blocks
){
    int block_id = blockIdx.x;
    int b = blockIdx.y;
    int tid = threadIdx.x;

    if (block_id >= num_blocks || b >= B || tid >= block_size)
        return;

    int row_block = row_idx[block_id];
    int col_block = col_idx[block_id];

    int row_start = row_block * block_size;
    int col_start = col_block * block_size;

    for (int k = 0; k < block_size; k++){
        float h_val = h[b * H + col_start + k];

        float w_val = W[(row_start + tid) * H + (col_start + k)];

        atomicAdd(
            &out[b * (4*H) + (row_start + tid)],
            h_val * w_val
        );
    }
}

torch::Tensor block_sparse_forward(
    torch::Tensor h,
    torch::Tensor W,
    torch::Tensor row_idx,
    torch::Tensor col_idx,
    int block_size
){
    int B = h.size(0);
    int H = h.size(1);
    int num_blocks = row_idx.size(0);

    auto out = torch::zeros({B, 4*H}, h.options());

    dim3 grid(num_blocks, B);
    dim3 block(block_size);

    block_sparse_matmul_kernel<<<grid, block>>>(
        h.data_ptr<float>(),
        W.data_ptr<float>(),
        row_idx.data_ptr<int>(),
        col_idx.data_ptr<int>(),
        out.data_ptr<float>(),
        B,
        H,
        block_size,
        num_blocks
    );

    return out;
}

Writing sparse_lstm_kernel.cu


In [88]:
%%writefile sparse_binding.cpp

#include <torch/extension.h>

torch::Tensor block_sparse_forward(
    torch::Tensor h,
    torch::Tensor W,
    torch::Tensor row_idx,
    torch::Tensor col_idx,
    int block_size
);

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &block_sparse_forward, "Block Sparse Forward");
}

Writing sparse_binding.cpp


In [90]:
# =========================
# COMPILE BLOCK-SPARSE CUDA (FIXED FULL BLOCK)
# =========================

# install ninja if missing (safe to run again)
!pip install ninja -q

from torch.utils.cpp_extension import load
import os

# clear old builds (VERY IMPORTANT)
!rm -rf /root/.cache/torch_extensions

# compile
sparse_cuda = load(
    name="sparse_cuda",
    sources=["sparse_binding.cpp", "sparse_lstm_kernel.cu"],
    verbose=True,
    extra_cuda_cflags=["-O3"],
    extra_cflags=["-O3"]
)

print("✅ sparse CUDA compiled successfully")

✅ sparse CUDA compiled successfully


In [91]:
# =========================
# TEST BLOCK-SPARSE CUDA KERNEL
# =========================

B = batch_size
H = hidden_size
block_size = 16

# dummy data
h = torch.randn(B, H, device=device).contiguous()
W = torch.randn(4*H, H, device=device).contiguous()

# fake indices (small test)
row_idx = torch.randint(0, 4*H//block_size, (50,), device=device, dtype=torch.int32)
col_idx = torch.randint(0, H//block_size, (50,), device=device, dtype=torch.int32)

out = sparse_cuda.forward(h, W, row_idx, col_idx, block_size)

print("sparse kernel output:", out.shape)

sparse kernel output: torch.Size([64, 2048])


In [92]:
# =========================
# CONVERT BLOCK MASK TO INDICES
# =========================

def mask_to_indices(mask):
    row_idx = []
    col_idx = []

    rows, cols = mask.shape

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] == 1:
                row_idx.append(i)
                col_idx.append(j)

    row_idx = torch.tensor(row_idx, dtype=torch.int32, device=device)
    col_idx = torch.tensor(col_idx, dtype=torch.int32, device=device)

    return row_idx, col_idx

In [93]:
# =========================
# FINAL BLOCK-SPARSE CUDA LSTM
# =========================

class SparseCUDALSTM(nn.Module):
    def __init__(self, input_size, hidden_size, sparsity=0.5, block_size=16):
        super().__init__()

        self.hidden_size = hidden_size
        self.block_size = block_size

        self.W_x = nn.Linear(input_size, 4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

        # create mask
        block_mask = create_block_mask(
            4 * hidden_size,
            hidden_size,
            block_size,
            sparsity
        )

        self.register_buffer("block_mask", block_mask)

        # convert to indices (ONCE)
        row_idx, col_idx = mask_to_indices(block_mask)

        self.register_buffer("row_idx", row_idx)
        self.register_buffer("col_idx", col_idx)

    def forward(self, x, hidden):
        B, T, D = x.shape
        H = self.hidden_size

        h, c = hidden

        # input GEMM
        x_reshaped = x.reshape(B * T, D)
        x_proj = self.W_x(x_reshaped)
        x_proj = x_proj.view(T, B, 4 * H).contiguous()

        outputs = []

        for t in range(T):
            # 🔥 CUDA SPARSE KERNEL
            h_proj = sparse_cuda.forward(
                h,
                self.W_h.weight,
                self.row_idx,
                self.col_idx,
                self.block_size
            ) + self.W_h.bias

            gates = x_proj[t] + h_proj

            h, c = fused_lstm_step(gates, c)

            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)

        return outputs, (h, c)

In [94]:
# =========================
# TEST FINAL SPARSE CUDA LSTM
# =========================

cuda_sparse_model = SparseCUDALSTM(
    input_size=hidden_size,
    hidden_size=hidden_size,
    sparsity=0.5,
    block_size=16
).to(device)

x, _ = get_batch(data, 0, seq_len)
x = model.embedding(x)

hidden = init_hidden_manual(batch_size, hidden_size, device)

out, hidden = cuda_sparse_model(x, hidden)

print("final sparse CUDA output:", out.shape)

final sparse CUDA output: torch.Size([64, 128, 512])


In [98]:
# =========================
# FINAL WORKING BENCHMARK (NO BUGS)
# =========================

def benchmark_cudnn(model, x_tokens, name, steps=50):
    model.eval()

    # WARMUP
    for _ in range(10):
        hidden = init_hidden(batch_size, hidden_size, num_layers, device)
        out, hidden = model(x_tokens, hidden)

    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    times = []

    for _ in range(steps):
        hidden = init_hidden(batch_size, hidden_size, num_layers, device)

        starter.record()
        out, hidden = model(x_tokens, hidden)
        ender.record()

        torch.cuda.synchronize()
        times.append(starter.elapsed_time(ender))

    avg_time = sum(times) / len(times)
    print(f"{name}: {avg_time:.3f} ms")

    return avg_time


def benchmark_cuda(model, x_embed, name, steps=50):
    model.eval()

    # WARMUP
    for _ in range(10):
        hidden = init_hidden_manual(batch_size, hidden_size, device)
        out, hidden = model(x_embed, hidden)

    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)

    times = []

    for _ in range(steps):
        hidden = init_hidden_manual(batch_size, hidden_size, device)

        starter.record()
        out, hidden = model(x_embed, hidden)
        ender.record()

        torch.cuda.synchronize()
        times.append(starter.elapsed_time(ender))

    avg_time = sum(times) / len(times)
    print(f"{name}: {avg_time:.3f} ms")

    return avg_time


# =========================
# PREP INPUTS
# =========================

x_tokens, _ = get_batch(data, 0, seq_len)   # int input
x_embed = model.embedding(x_tokens)         # float input


# =========================
# RUN BENCHMARK
# =========================

print("===== BENCHMARK RESULTS =====")

dense_time = benchmark_cudnn(model, x_tokens, "cuDNN LSTM")
cuda_dense_time = benchmark_cuda(cuda_model, x_embed, "CUDA Dense LSTM")
sparse_time = benchmark_cuda(cuda_sparse_model, x_embed, "Sparse CUDA LSTM")

print("\n===== SPEEDUP =====")

print(f"Sparse vs cuDNN: {dense_time / sparse_time:.2f}x")
print(f"Sparse vs CUDA Dense: {cuda_dense_time / sparse_time:.2f}x")

===== BENCHMARK RESULTS =====
cuDNN LSTM: 27.216 ms
CUDA Dense LSTM: 35.914 ms
Sparse CUDA LSTM: 206.257 ms

===== SPEEDUP =====
Sparse vs cuDNN: 0.13x
Sparse vs CUDA Dense: 0.17x
